## Breast Cancer Survival Analysis (METABRIC)

### Objective
To analyze survival outcomes of breast cancer patients using clinical features  
and identify key risk factors associated with mortality.

---

### Research Questions

- Which clinical factors significantly affect patient survival?
- How do tumor characteristics influence mortality risk?
- Can we quantify patient risk using survival analysis models?

---

### Methodology

- Survival Analysis (Kaplan-Meier)
- Cox Proportional Hazards Model
- Feature impact interpretation (hazard ratio)

In [ ]:
1. 라이브러리 import 

import pandas as pd
import numpy as np

# display setting
pd.set_option("display.max_columns", 100)

In [7]:
# 2. 데이터 로드 

import pandas as pd

file_path = "/Users/junyeong/Desktop/Projects/public/grad-portfolio-ml/projects/survival-analysis-breast-cancer/data/raw/brca_metabric_clinical_data.tsv"

print("Shape:", df.shape)
df.head()

Shape: (2509, 39)


,Study ID,Patient ID,Sample ID,Age at Diagnosis,Type of Breast Surgery,Cancer Type,Cancer Type Detailed,Cellularity,Chemotherapy,Pam50 + Claudin-low subtype,Cohort,ER status measured by IHC,ER Status,Neoplasm Histologic Grade,HER2 status measured by SNP6,HER2 Status,Tumor Other Histologic Subtype,Hormone Therapy,Inferred Menopausal State,Integrative Cluster,Primary Tumor Laterality,Lymph nodes examined positive,Mutation Count,Nottingham prognostic index,Oncotree Code,Overall Survival (Months),Overall Survival Status,PR Status,Radio Therapy,Relapse Free Status (Months),Relapse Free Status,Number of Samples Per Patient,Sample Type,Sex,3-Gene classifier subtype,TMB (nonsynonymous),Tumor Size,Tumor Stage,Patient's Vital Status
0,brca_metabric,MB-0000,MB-0000,75.65,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,NaN,NO,claudin-low,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,YES,Post,4ER+,Right,10.0,NaN,6.044,IDC,140.500000,0:LIVING,Negative,YES,140.500000,0:Not Recurred,1,Primary,Female,ER-/HER2-,0.000000,22.0,2.0,Living
1,brca_metabric,MB-0002,MB-0002,43.19,BREAST CONSERVING,Breast Cancer,Breast Invasive Ductal Carcinoma,High,NO,LumA,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Ductal/NST,YES,Pre,4ER+,Right,0.0,2.0,4.020,IDC,84.633333,0:LIVING,Positive,YES,84.633333,0:Not Recurred,1,Primary,Female,ER+/HER2- High Prolif,2.615035,10.0,1.0,Living
2,brca_metabric,MB-0005,MB-0005,48.87,MASTECTOMY,Breast Cancer,Breast Invasive Ductal Carcinoma,High,YES,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Ductal/NST,YES,Pre,3,Right,1.0,2.0,4.030,IDC,163.700000,1:DECEASED,Positive,NO,153.300000,1:Recurred,1,Primary,Female,NaN,2.615035,15.0,2.0,Died of Disease
3,brca_metabric,MB-0006,MB-0006,47.68,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,Moderate,YES,LumB,1.0,Positve,Positive,2.0,NEUTRAL,Negative,Mixed,YES,Pre,9,Right,3.0,1.0,4.050,MDLC,164.933333,0:LIVING,Positive,YES,164.933333,0:Not Recurred,1,Primary,Female,NaN,1.307518,25.0,2.0,Living
4,brca_metabric,MB-0008,MB-0008,76.97,MASTECTOMY,Breast Cancer,Breast Mixed Ductal and Lobular Carcinoma,High,YES,LumB,1.0,Positve,Positive,3.0,NEUTRAL,Negative,Mixed,YES,Post,9,Right,8.0,2.0,6.080,MDLC,41.366667,1:DECEASED,Positive,YES,18.800000,1:Recurred,1,Primary,Female,ER+/HER2- High Prolif,2.615035,40.0,2.0,Died of Disease


In [9]:
# 3. 컬럼 확인

df.columns.tolist()

['Study ID',
 'Patient ID',
 'Sample ID',
 'Age at Diagnosis',
 'Type of Breast Surgery',
 'Cancer Type',
 'Cancer Type Detailed',
 'Cellularity',
 'Chemotherapy',
 'Pam50 + Claudin-low subtype',
 'Cohort',
 'ER status measured by IHC',
 'ER Status',
 'Neoplasm Histologic Grade',
 'HER2 status measured by SNP6',
 'HER2 Status',
 'Tumor Other Histologic Subtype',
 'Hormone Therapy',
 'Inferred Menopausal State',
 'Integrative Cluster',
 'Primary Tumor Laterality',
 'Lymph nodes examined positive',
 'Mutation Count',
 'Nottingham prognostic index',
 'Oncotree Code',
 'Overall Survival (Months)',
 'Overall Survival Status',
 'PR Status',
 'Radio Therapy',
 'Relapse Free Status (Months)',
 'Relapse Free Status',
 'Number of Samples Per Patient',
 'Sample Type',
 'Sex',
 '3-Gene classifier subtype',
 'TMB (nonsynonymous)',
 'Tumor Size',
 'Tumor Stage',
 "Patient's Vital Status"]

In [15]:
# 4. survival 변수 정의

# event 정의
df["event"] = df["Patient's Vital Status"].apply(
    lambda x: 1 if x == "Died of Disease" else 0
)

# time 정의
df["time"] = pd.to_numeric(df["Overall Survival (Months)"], errors="coerce")

df[["Patient's Vital Status", "event", "Overall Survival (Months)", "time"]].head()

,Patient's Vital Status,event,Overall Survival (Months),time
0,Living,0,140.500000,140.500000
1,Living,0,84.633333,84.633333
2,Died of Disease,1,163.700000,163.700000
3,Living,0,164.933333,164.933333
4,Died of Disease,1,41.366667,41.366667


In [ ]:
# 5. 사용할 feature 선택

selected_features = [
    "Age at Diagnosis",
    "Neoplasm Histologic Grade",
    "Tumor Stage",
    "ER Status",
    "HER2 Status",
    "Lymph nodes examined positive",
    "Tumor Size",
    "time",
    "event",
]

df_model = df[selected_features].copy()
df_model = df_model.dropna()

print("Final dataset shape:", df_model.shape)
df_model.head()

Final dataset shape: (1354, 9)


,Age at Diagnosis,Neoplasm Histologic Grade,Tumor Stage,ER Status,HER2 Status,Lymph nodes examined positive,Tumor Size,time,event
0,75.65,3.0,2.0,Positive,Negative,10.0,22.0,140.500000,0
1,43.19,3.0,1.0,Positive,Negative,0.0,10.0,84.633333,0
2,48.87,2.0,2.0,Positive,Negative,1.0,15.0,163.700000,1
3,47.68,2.0,2.0,Positive,Negative,3.0,25.0,164.933333,0
4,76.97,3.0,2.0,Positive,Negative,8.0,40.0,41.366667,1


In [17]:
# 5. 데이터 타입 확인

df_model.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1354 entries, 0 to 1743
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age at Diagnosis               1354 non-null   float64
 1   Neoplasm Histologic Grade      1354 non-null   float64
 2   Tumor Stage                    1354 non-null   float64
 3   ER Status                      1354 non-null   object 
 4   HER2 Status                    1354 non-null   object 
 5   Lymph nodes examined positive  1354 non-null   float64
 6   Tumor Size                     1354 non-null   float64
 7   time                           1354 non-null   float64
 8   event                          1354 non-null   int64  
dtypes: float64(6), int64(1), object(2)
memory usage: 105.8+ KB


In [18]:
# 6. event 분포 확인

df_model["event"].value_counts()

event
0    896
1    458
Name: count, dtype: int64

In [19]:
# 7. survival time 확인

df_model["time"].describe()

count    1354.000000
mean      127.136928
std        78.024751
min         0.100000
25%        61.500000
50%       116.783333
75%       187.908333
max       351.000000
Name: time, dtype: float64

## Target Definition

- **Time variable**: `Overall Survival (Months)`
- **Event variable**: derived from `Patient's Vital Status`
  - `1` = Died of Disease
  - `0` = Living

## Selected Clinical Features

- Age at Diagnosis
- Neoplasm Histologic Grade
- Tumor Stage
- ER Status
- HER2 Status
- Lymph nodes examined positive
- Tumor Size